## Remarks: I have used Claude for idea generation

# Homework 4 — Method Report
A concise breakdown of every method in the `Homework4` class.

## 1. `randomQuickSort(nums)`
Public entry point. Copies the input list to avoid mutation, calls the recursive helper, and returns the sorted copy.

In [ ]:
def randomQuickSort(self, nums: list) -> list:
    result = nums[:]
    self._quick_sort_helper(result, 0, len(result) - 1)
    return result

## 2. `_quick_sort_helper(seq, lo, hi)`
Recursive controller. Base case: `lo >= hi` (segment has 0 or 1 elements). Otherwise partitions around a pivot and recurses on both halves.

In [ ]:
def _quick_sort_helper(self, seq, lo, hi):
    if lo >= hi:
        return
    split_pos = self._partition_random(seq, lo, hi)
    self._quick_sort_helper(seq, lo, split_pos - 1)
    self._quick_sort_helper(seq, split_pos + 1, hi)

## 3. `_partition_random(seq, lo, hi)`
Core partitioning logic:
1. **Random pivot** — pick a random index, swap that element to `lo` (front).
2. **Right-to-left scan** — walk a `cursor` from `hi` down to `lo+1`. Any element ≥ pivot gets swapped toward the right via a shrinking `wall` pointer.
3. **Place pivot** — decrement `wall` once more and swap the pivot from `lo` into `wall`. Return `wall` as the pivot’s final index.

This mirror of the classic Lomuto scheme still runs in O(n) per call, and the random pivot choice keeps expected total work at O(n log n).

In [ ]:
def _partition_random(self, seq, lo, hi):
    chosen = random.randint(lo, hi)
    seq[lo], seq[chosen] = seq[chosen], seq[lo]

    pivot_val = seq[lo]
    wall = hi + 1

    for cursor in range(hi, lo, -1):
        if seq[cursor] >= pivot_val:
            wall -= 1
            seq[cursor], seq[wall] = seq[wall], seq[cursor]

    wall -= 1
    seq[lo], seq[wall] = seq[wall], seq[lo]
    return wall

## 4. `heapSort(nums)`
Two-phase driver:
- **Phase 1 (Build)** — iterate from the last internal node (`length//2 - 1`) down to 0, calling `_heapify` to establish the max-heap property bottom-up.
- **Phase 2 (Extract)** — repeatedly swap the root (max) with the last unsorted element, shrink the heap by one, and re-heapify the root.

In [ ]:
def heapSort(self, nums: list) -> list:
    data = nums[:]
    length = len(data)

    k = length // 2 - 1
    while k >= 0:
        self._heapify(data, length, k)
        k -= 1

    end = length - 1
    while end > 0:
        data[0], data[end] = data[end], data[0]
        self._heapify(data, end, 0)
        end -= 1

    return data

## 5. `_heapify(data, size, root)`
Iterative sift-down. Starting at `root`, compare with left child (`2*pos+1`) and right child (`2*pos+2`). If the largest of the three isn't the parent, swap and continue downward. Loop exits when the parent is already the largest — heap property restored.

Using a `while` loop instead of recursion avoids extra stack frames, which matters at large N.

In [ ]:
def _heapify(self, data, size, root):
    pos = root
    while True:
        lc = 2 * pos + 1
        rc = 2 * pos + 2
        top = pos

        if lc < size and data[lc] > data[top]:
            top = lc
        if rc < size and data[rc] > data[top]:
            top = rc

        if top == pos:
            break

        data[pos], data[top] = data[top], data[pos]
        pos = top

## 6. `run_performance_analysis()`
Benchmarks both sorting methods across input sizes `[10k, 100k, 500k, 1M]`. For each size and algorithm it:
1. Generates random floats in [-1000, 1000].
2. Records wall-clock time (`time.perf_counter`) and peak memory (`tracemalloc`).
3. Catches `RecursionError` gracefully (relevant for deep quicksort stacks at large N).

In [ ]:
def run_performance_analysis(self):
    sizes = [10000, 100000, 500000, 1000000]
    for n in sizes:
        data = [random.uniform(-1000.0, 1000.0) for _ in range(n)]
        for algo_name, algo_func in [("QuickSort", self.randomQuickSort), ("HeapSort", self.heapSort)]:
            tracemalloc.start()
            start_time = time.perf_counter()
            try:
                algo_func(list(data))
                duration = time.perf_counter() - start_time
                _, peak_mem = tracemalloc.get_traced_memory()
                print(f"{algo_name:10} | N={n} | Time: {duration:.4f}s | Mem: {peak_mem/1e6:.2f} MB")
            except RecursionError:
                print(f"{algo_name:10} | N={n} | RecursionError")
            finally:
                tracemalloc.stop()